In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch

# Load the dataset from the raw data folder
# Note: Since the image showed an Excel icon, change '.csv' to '.xlsx' and use pd.read_excel() if it is an Excel file.
file_path = '../data/raw/Modified_SQL_Dataset.csv'
df = pd.read_csv(file_path)

# Display the first 5 rows to understand the structure
display(df.head())

,Query,Label
0,""" or pg_sleep ( __TIME__ ) --",1
1,create user name identified by pass123 tempora...,1
2,AND 1 = utl_inaddr.get_host_address ( ...,1
3,select * from users where id = '1' or @ @1 ...,1
4,"select * from users where id = 1 or 1#"" ( ...",1


In [6]:
# Check the total number of rows and columns
print(f"Dataset Shape: {df.shape}\n")

# Check for any missing or corrupted rows in the dataset itself
print("Missing Values per Column:")
print(df.isnull().sum())
print("\n")

# Check the balance of normal vs malicious queries
# Note: Update 'Label' if your column has a different name (e.g., 'Class', 'Is_Malicious')
if 'Label' in df.columns:
    print("Label Distribution:")
    print(df['Label'].value_counts())
else:
    print("Columns in dataset:", df.columns)

Dataset Shape: (30919, 2)

Missing Values per Column:
Query    0
Label    0
dtype: int64


Label Distribution:
Label
0    19537
1    11382
Name: count, dtype: int64


In [7]:
# Drop any rows with missing values
df = df.dropna()

text_column = 'Query'
label_column = 'Label'
df[text_column] = df[text_column].astype(str)

X = df[text_column].values
y = df[label_column].values

# First Split: Separate out 15% for the final Test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Second Split: Split the remaining 85% into Training (70% overall) and Validation (15% overall)
# 0.15 / 0.85 is roughly 0.176
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 21655
Validation samples: 4626
Testing samples: 4638


In [8]:
import re
from collections import Counter

# 1. Simple text cleaning and tokenization function
def tokenize_sql(query):
    # Convert to lowercase and add spaces around punctuation so they become distinct tokens
    query = str(query).lower()
    query = re.sub(r'([^\w\s])', r' \1 ', query)
    return query.split()

# 2. Build the vocabulary based ONLY on the training data
vocab = Counter()
for query in X_train:
    vocab.update(tokenize_sql(query))

# Create a mapping from word to integer (0 is reserved for padding, 1 for unknown words)
word2idx = {word: idx + 2 for idx, (word, count) in enumerate(vocab.items())}
word2idx['<PAD>'] = 0
word2idx['<UNK>'] = 1

def encode_query(query, max_length=100):
    tokens = tokenize_sql(query)
    # Convert tokens to integers, use <UNK> if word is not in vocab
    seq = [word2idx.get(word, word2idx['<UNK>']) for word in tokens]
    
    # Pad or truncate the sequence to max_length
    if len(seq) < max_length:
        seq += [word2idx['<PAD>']] * (max_length - len(seq))
    else:
        seq = seq[:max_length]
    return seq

# ... (Keep the tokenization and word2idx logic the same) ...

max_seq_length = 100
X_train_encoded = np.array([encode_query(q, max_seq_length) for q in X_train])
X_val_encoded   = np.array([encode_query(q, max_seq_length) for q in X_val])
X_test_encoded  = np.array([encode_query(q, max_seq_length) for q in X_test])

print(f"Vocabulary Size: {len(word2idx)}")
print(f"Sample encoded query shape: {X_train_encoded[0].shape}")
print(f"First encoded query:\n{X_train_encoded[0]}")

Vocabulary Size: 19175
Sample encoded query shape: (100,)
First encoded query:
[ 2  3  4  5  5  5  6  7  8  9  8 10 11 12  8 13 11  8 14  8 15 11 16  8
 17  5  5  5  5  5 11  2  5 18  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0]


In [9]:
from torch.utils.data import TensorDataset, DataLoader

# Convert numpy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train_encoded, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val_encoded, dtype=torch.long) # NEW
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)      # NEW

X_test_tensor = torch.tensor(X_test_encoded, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)      # NEW
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
val_loader = DataLoader(val_dataset, shuffle=False, batch_size=batch_size)   # NEW
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of testing batches: {len(test_loader)}")

Number of training batches: 677
Number of validation batches: 145
Number of testing batches: 145


In [10]:
import sys
import os

# Add the parent directory (project root) to Python's path
sys.path.append(os.path.abspath('../'))

# Now try importing again to verify it works
from src.models.sql_models import SQLCNN, SQLLSTM
print("Modules imported successfully!")

Modules imported successfully!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from src.models.sql_models import SQLCNN, SQLLSTM

# 1. Detect Hardware Accelerator
# Uses NVIDIA CUDA if available, otherwise falls back to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training will run on: {device}")

def train_and_evaluate(model, train_loader, val_loader, model_name="model", epochs=50):
    # 2. Send the model to the GPU
    model = model.to(device)
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_loss = float('inf')
    best_model_weights = None
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        
        for batch_x, batch_y in train_loader:
            # 3. Send the data batches to the GPU
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            predictions = model(batch_x).squeeze(1)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                # 4. Send validation data to GPU
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                
                predictions = model(batch_x).squeeze(1)
                loss = criterion(predictions, batch_y)
                val_loss += loss.item()
                
                rounded_preds = torch.round(torch.sigmoid(predictions))
                correct += (rounded_preds == batch_y).sum().item()
                total += batch_y.size(0)
                
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = correct / total
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
            torch.save(best_model_weights, f'{model_name}_best.pth')
            print(f"--> New best model saved as {model_name}_best.pth")

# Initialize and train Model 1 (CNN)
print("\nTraining Conv1D CNN...")
cnn_model = SQLCNN(vocab_size=len(word2idx))
train_and_evaluate(cnn_model, train_loader, val_loader, model_name="SQL_CNN", epochs=10)

print("\n-----------------------------------\n")

# Initialize and train Model 2 (LSTM)
print("Training Bidirectional LSTM...")
lstm_model = SQLLSTM(vocab_size=len(word2idx))
train_and_evaluate(lstm_model, train_loader, val_loader, model_name="SQL_LSTM", epochs=10)

Training will run on: cuda

Training Conv1D CNN...
Epoch 1/10 | Train Loss: 0.0589 | Val Loss: 0.0060 | Val Acc: 0.9981
--> New best model saved as SQL_CNN_best.pth
Epoch 2/10 | Train Loss: 0.0147 | Val Loss: 0.0054 | Val Acc: 0.9987
--> New best model saved as SQL_CNN_best.pth
Epoch 3/10 | Train Loss: 0.0116 | Val Loss: 0.0064 | Val Acc: 0.9987
Epoch 4/10 | Train Loss: 0.0093 | Val Loss: 0.0050 | Val Acc: 0.9987
--> New best model saved as SQL_CNN_best.pth
Epoch 5/10 | Train Loss: 0.0062 | Val Loss: 0.0064 | Val Acc: 0.9985
Epoch 6/10 | Train Loss: 0.0057 | Val Loss: 0.0112 | Val Acc: 0.9976
Epoch 7/10 | Train Loss: 0.0076 | Val Loss: 0.0044 | Val Acc: 0.9989
--> New best model saved as SQL_CNN_best.pth
Epoch 8/10 | Train Loss: 0.0054 | Val Loss: 0.0149 | Val Acc: 0.9983
Epoch 9/10 | Train Loss: 0.0028 | Val Loss: 0.0095 | Val Acc: 0.9987
Epoch 10/10 | Train Loss: 0.0042 | Val Loss: 0.0116 | Val Acc: 0.9987

-----------------------------------

Training Bidirectional LSTM...
Epoch 1/1

In [12]:
import json
import os

# Ensure the directory exists
os.makedirs('../data/processed', exist_ok=True)

# Save the vocabulary dictionary as a JSON file
vocab_path = '../data/processed/sql_word2idx.json'
with open(vocab_path, 'w') as f:
    json.dump(word2idx, f)

print(f"Vocabulary successfully saved to {vocab_path}")

Vocabulary successfully saved to ../data/processed/sql_word2idx.json
